# Protein Sequence Mapping

This notebook maps protein IDs from your metabolic model to NCBI protein accessions using BLASTp. The output CSV is used in **Step 2** of the Universal Metabolic Community Pipeline.

---

## How to use this notebook

1. **Run Cell 1** — installs BLAST automatically (takes ~1 minute)
2. **Run Cell 2** — upload your FASTA files when prompted
3. **Run Cell 3** — runs BLAST and generates your mapping CSV
4. **Download the CSV** — find it in the Files panel on the left, right-click and download
5. **Upload the CSV** to Step 2 of the pipeline

> To run a cell: click on it and press **Shift + Enter**, or click the ▶ button on the left of the cell.

## Cell 1 — Install BLAST and dependencies
Run this first. It installs everything needed. This takes about 1 minute.

In [ ]:
import shutil
if not shutil.which('blastp'):
    import subprocess
    print('Installing BLAST...')
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ncbi-blast+'], check=True)
    print('BLAST installed.')
else:
    print('BLAST already installed.')

%pip install -q pandas

## Cell 2 — Upload your FASTA files

When you run this cell, two upload prompts will appear — one for each file:

- **Query FASTA**: the `.faa` protein file from your metabolic model
- **Database FASTA**: the `.faa` protein file downloaded from NCBI for the same species

Repeat this for each species you want to process.

In [ ]:
from google.colab import files
import os

print('Upload your QUERY FASTA file (.faa) — protein sequences from your metabolic model')
uploaded_query = files.upload()
query_filename_raw = list(uploaded_query.keys())[0]
query_filename = query_filename_raw.replace(' ', '_')
if query_filename != query_filename_raw:
    os.rename(query_filename_raw, query_filename)
print(f'Query file: {query_filename}')

print('\nUpload your DATABASE FASTA file (.faa) — protein sequences downloaded from NCBI')
uploaded_db = files.upload()
db_filename_raw = list(uploaded_db.keys())[0]
db_filename = db_filename_raw.replace(' ', '_')
if db_filename != db_filename_raw:
    os.rename(db_filename_raw, db_filename)
print(f'Database file: {db_filename}')

# Derive species name from query filename
species_name = query_filename.split('_')[0]
print(f'\nSpecies prefix detected: {species_name}')

## Cell 3 — Run BLAST and generate the mapping CSV

This cell runs BLASTp and filters the results. It may take a few minutes depending on the size of your files.

When done, your mapping CSV will appear in the **Files panel** on the left (📁). Right-click it and select **Download**.

In [ ]:
import subprocess
import pandas as pd
import os

db_name = f'{species_name}_db'
blast_output = f'{species_name}_blast.txt'
output_csv = f'{species_name}_protein_id_mapping.csv'

# Build BLAST database
print('Building BLAST database...')
subprocess.run(['makeblastdb', '-in', db_filename, '-dbtype', 'prot', '-out', db_name],
               check=True, capture_output=True)

# Run BLASTp
print('Running BLASTp... (this may take a few minutes)')
blast_cmd = [
    'blastp', '-query', query_filename, '-db', db_name,
    '-out', blast_output,
    '-outfmt', '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs',
    '-num_threads', '2'
]
subprocess.run(blast_cmd, check=True, capture_output=True)
print('BLAST complete.')

# Load results
cols = ['qseqid','sseqid','pident','length','mismatch','gapopen','qstart','qend','sstart','send','evalue','bitscore','qcovs']
blast_df = pd.read_csv(blast_output, sep='\t', header=None, names=cols)

# Load database sequence lengths
db_lengths = {}
current_id = None
current_len = 0
with open(db_filename) as f:
    for line in f:
        if line.startswith('>'):
            if current_id: db_lengths[current_id] = current_len
            current_id = line[1:].strip().split()[0]
            current_len = 0
        else:
            current_len += len(line.strip())
    if current_id: db_lengths[current_id] = current_len

# Load query sequence lengths
query_lengths = {}
current_id = None
current_len = 0
with open(query_filename) as f:
    for line in f:
        if line.startswith('>'):
            if current_id: query_lengths[current_id] = current_len
            current_id = line[1:].strip().split()[0]
            current_len = 0
        else:
            current_len += len(line.strip())
    if current_id: query_lengths[current_id] = current_len

# Calculate coverages and filter
blast_df['qlen'] = blast_df['qseqid'].map(query_lengths)
blast_df['slen'] = blast_df['sseqid'].map(db_lengths)
blast_df['qcoverage'] = (blast_df['qend'] - blast_df['qstart'] + 1) / blast_df['qlen']
blast_df['scoverage'] = blast_df['length'] / blast_df['slen']

filtered = blast_df[
    (blast_df['pident'] >= 95) &
    (blast_df['gapopen'] == 0) &
    (blast_df['mismatch'] <= 1) &
    (blast_df['qcoverage'] >= 0.85) &
    (blast_df['scoverage'] >= 0.85)
]

# Keep best hit per query
best = filtered.sort_values('evalue').groupby('qseqid').first().reset_index()
result = best[['qseqid', 'sseqid']].rename(columns={'qseqid': 'identifier_query', 'sseqid': 'identifier_db'})
result.to_csv(output_csv, index=False)

print(f'\nDone! {len(result)} matches found out of {blast_df["qseqid"].nunique()} query sequences.')
print(f'Output saved as: {output_csv}')
print('\nFind the file in the Files panel on the left (📁), right-click it and select Download.')